# NFL Draft Prediction — GCI World 2026

## Competition Overview

The **NFL Draft** is the annual event where 32 teams select college players to join as professionals. Being drafted means a player has been recognized as talented enough to compete at the highest level.

### Objective
Build a **binary classification model** that predicts whether a college player will be selected (**Drafted = 1**) or not (**Drafted = 0**) using physical performance tests, position, body measurements, and school data.

### Evaluation Metric
Predictions are evaluated with **AUC** (Area Under the ROC Curve). Goal: **maximize AUC**.

### Dataset
| File | Description |
|---|---|
| `train.csv` | Training data (2,781 rows, 16 columns) |
| `test.csv` | Test data (696 rows, 15 columns — no `Drafted` column) |
| `sample_submission.csv` | Required submission format |

### Features
| Variable | Definition |
|---|---|
| `Id` | Unique player ID |
| `Year` | Record year |
| `Age` | Player age |
| `School` | College |
| `Height` | Height |
| `Weight` | Weight |
| `Sprint_40yd` | 40-yard dash time (lower = faster) |
| `Vertical_Jump` | Vertical jump height |
| `Bench_Press_Reps` | Bench press repetitions |
| `Broad_Jump` | Broad jump distance |
| `Agility_3cone` | 3-cone drill time |
| `Shuttle` | 20-yard shuttle time |
| `Player_Type` | Broad player role category |
| `Position_Type` | Position type |
| `Position` | Specific position |
| `Drafted` | **Target**: 1 = drafted, 0 = not drafted |

---
## 1. Setup

In [ ]:
!pip install lightgbm xgboost optuna -q


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
try:
    import lightgbm as lgb
    LGBM_AVAILABLE = True
except ImportError:
    LGBM_AVAILABLE = False
    print("LightGBM not available.")

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print("XGBoost not available.")

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, roc_curve, auc

SEED = 42
np.random.seed(SEED)

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_theme(style='whitegrid', palette='muted')

print("Libraries imported successfully")


---
## 2. Load Data

> **Note:** Set `PATH` to point to the `input/` folder containing `train.csv`, `test.csv`, and `sample_submission.csv`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/MyDrive/GCI World - UTokio/competition"

PATH = Path("input")

train = pd.read_csv(PATH / 'train.csv')
test  = pd.read_csv(PATH / 'test.csv')
sample_sub = pd.read_csv(PATH / 'sample_submission.csv')

print(f"Train: {train.shape[0]} rows, {train.shape[1]} columns")
print(f"Test:  {test.shape[0]} rows, {test.shape[1]} columns")
print(f"Sample submission: {sample_sub.shape}")


In [ ]:
train.head()

In [ ]:
train.info()

In [ ]:
train.describe().round(2)

---
## 3. Exploratory Data Analysis (EDA)

### 3.1 Target Variable Distribution

In [ ]:
counts = train['Drafted'].value_counts()
pcts   = train['Drafted'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(['Not Drafted (0)', 'Drafted (1)'], counts.values, color=['#E74C3C', '#2ECC71'])
axes[0].set_title('Drafted Distribution', fontsize=14)
axes[0].set_ylabel('Player count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontsize=12, fontweight='bold')

axes[1].pie(pcts.values, labels=[f'Not Drafted ({pcts[0]:.1f}%)', f'Drafted ({pcts[1]:.1f}%)'],
            colors=['#E74C3C', '#2ECC71'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Drafted Proportion', fontsize=14)

plt.suptitle('Target Variable: Drafted', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Class counts: {dict(counts)}")
print("Dataset is moderately imbalanced (~65% drafted, ~35% not drafted).")

### 3.2 Missing Values

In [ ]:
def missing_report(df, name):
    nulls = df.isnull().sum()
    pct   = (nulls / len(df) * 100).round(2)
    report = pd.DataFrame({'Nulls': nulls, '% Missing': pct})
    report = report[report['Nulls'] > 0].sort_values('% Missing', ascending=False)
    print(f"\n=== {name} ===")
    print(report if len(report) > 0 else "  No missing values.")
    return report

missing_train = missing_report(train, "Train")
missing_test  = missing_report(test, "Test")

In [ ]:
if len(missing_train) > 0:
    fig, ax = plt.subplots(figsize=(10, 4))
    missing_train['% Missing'].plot(kind='bar', ax=ax, color='#E74C3C')
    ax.set_title('Missing Values (%) — Train', fontsize=14)
    ax.set_ylabel('% Missing')
    ax.set_xlabel('Feature')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

### 3.3 Numeric Feature Distributions

In [ ]:
num_cols = train.select_dtypes(include='number').columns.tolist()
num_cols = [c for c in num_cols if c not in ['Id', 'Drafted']]

n_cols = 3
n_rows = (len(num_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    for label, color in [(0, '#E74C3C'), (1, '#2ECC71')]:
        subset = train[train['Drafted'] == label][col].dropna()
        axes[i].hist(subset, bins=30, alpha=0.5, color=color,
                     label=f'Drafted={label}', edgecolor='white')
    axes[i].set_title(col, fontsize=12)
    axes[i].legend(fontsize=9)
    axes[i].set_xlabel(col)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numeric Feature Distributions by Drafted', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.4 Correlation Heatmap

In [ ]:
# Sprint_40yd has negative correlation with Broad_Jump because lower time = faster sprint.
corr_cols = [c for c in train.select_dtypes(include='number').columns if c not in ['Id']]
corr = train[corr_cols].corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, square=True, linewidths=0.5, mask=mask)
plt.title('Correlation Heatmap', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nCorrelation with 'Drafted':")
print(corr['Drafted'].sort_values(ascending=False).to_string())

### 3.5 Categorical Features

In [ ]:
cat_cols = ['Player_Type', 'Position_Type', 'Position']

fig, axes = plt.subplots(1, len(cat_cols), figsize=(18, 6))

for i, col in enumerate(cat_cols):
    mean_drafted = train.groupby(col)['Drafted'].mean().sort_values(ascending=False)
    mean_drafted.plot(kind='bar', ax=axes[i], color='#3498DB', edgecolor='white')
    axes[i].set_title(f'Draft Rate by {col}', fontsize=12)
    axes[i].set_ylabel('Mean Draft Rate')
    axes[i].set_ylim(0, 1)
    axes[i].axhline(train['Drafted'].mean(), color='red', linestyle='--', alpha=0.7,
                    label=f'Global mean ({train["Drafted"].mean():.2f})')
    axes[i].legend(fontsize=9)
    axes[i].tick_params(axis='x', rotation=45)

plt.suptitle('Draft Selection Rate by Category', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Top 20 schools by draft rate (minimum 10 players)
top_schools = (train.groupby('School')['Drafted']
               .agg(['mean', 'count'])
               .rename(columns={'mean': 'Draft_Rate', 'count': 'Total_Players'})
               .query('Total_Players >= 10')
               .sort_values('Draft_Rate', ascending=False)
               .head(20))

plt.figure(figsize=(12, 6))
plt.barh(top_schools.index, top_schools['Draft_Rate'], color='#9B59B6')
plt.xlabel('Draft Rate')
plt.title('Top 20 Schools by Draft Rate (min. 10 players)', fontsize=13)
plt.axvline(train['Drafted'].mean(), color='red', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

### 3.6 Boxplots: Performance Features vs Drafted

In [ ]:
perf_cols = ['Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps',
             'Broad_Jump', 'Agility_3cone', 'Shuttle']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(perf_cols):
    data_plot = train[['Drafted', col]].dropna().copy()
    data_plot['Drafted_str'] = data_plot['Drafted'].map({0: 'Not Drafted', 1: 'Drafted'})
    sns.boxplot(x='Drafted_str', y=col, data=data_plot,
                palette={'Not Drafted': '#E74C3C', 'Drafted': '#2ECC71'}, ax=axes[i])
    axes[i].set_title(f'{col} by Drafted', fontsize=11)
    axes[i].set_xlabel('')

plt.suptitle('Performance Features vs Drafted', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Drafted players tend to have:")
print("  - LOWER Sprint_40yd, Agility_3cone, Shuttle (faster)")
print("  - HIGHER Vertical_Jump, Bench_Press_Reps, Broad_Jump (more athletic)")

---
## 4. Preprocessing

In [ ]:
train_proc = train.copy()
test_proc  = test.copy()

test_ids = test_proc['Id'].values

train_proc = train_proc.drop(columns=['Id'])
test_proc  = test_proc.drop(columns=['Id'])

print(f"Shapes — Train: {train_proc.shape} | Test: {test_proc.shape}")

### 4.1 Feature Engineering

In [ ]:
for df in [train_proc, test_proc]:
    df['BMI'] = df['Weight'] / (df['Height'] ** 2)
    df['Power_Index'] = df['Vertical_Jump'] * df['Broad_Jump']
    # Heavier players who run fast are especially valuable
    df['Speed_per_Weight'] = df['Weight'] / (df['Sprint_40yd'] + 1e-6)
    df['Agility_Combined'] = df['Agility_3cone'] + df['Shuttle']

    # Missing value indicators — may carry predictive signal
    for col in ['Age', 'Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps',
                'Broad_Jump', 'Agility_3cone', 'Shuttle']:
        df[f'{col}_missing'] = df[col].isnull().astype(int)

print(f"Feature engineering done — Train: {train_proc.shape} | Test: {test_proc.shape}")

### 4.2 Imputation

In [ ]:
# Impute with train median only (no data leakage)
cols_numeric = train_proc.select_dtypes(include='number').columns.tolist()
cols_to_impute = [c for c in cols_numeric if 'missing' not in c and c != 'Drafted']

train_medians = train_proc[cols_to_impute].median()

train_proc[cols_to_impute] = train_proc[cols_to_impute].fillna(train_medians)
test_proc[cols_to_impute]  = test_proc[cols_to_impute].fillna(train_medians)

print(f"Remaining nulls — Train: {train_proc.isnull().sum().sum()} | Test: {test_proc.isnull().sum().sum()}")

### 4.3 Categorical Encoding

In [ ]:
# Label encoding for position features
label_encoders = {}
for col in ['Player_Type', 'Position_Type', 'Position']:
    le = LabelEncoder()
    train_proc[col] = le.fit_transform(train_proc[col].astype(str))
    test_col = test_proc[col].astype(str).copy()
    test_col[~test_col.isin(le.classes_)] = le.classes_[0]
    test_proc[col] = le.transform(test_col)
    label_encoders[col] = le

# Target encoding for School (avoids 200+ one-hot columns)
school_draft_rate = train_proc.groupby('School')['Drafted'].mean()
global_mean = train_proc['Drafted'].mean()

train_proc['School_encoded'] = train_proc['School'].map(school_draft_rate).fillna(global_mean)
test_proc['School_encoded']  = test_proc['School'].map(school_draft_rate).fillna(global_mean)

train_proc = train_proc.drop(columns=['School'])
test_proc  = test_proc.drop(columns=['School'])

print(f"Encoding done — Train: {train_proc.shape} | Test: {test_proc.shape}")

In [ ]:
X = train_proc.drop(columns=['Drafted'])
y = train_proc['Drafted']
X_test = test_proc.copy()

print(f"X: {X.shape} | y: {y.shape} | X_test: {X_test.shape}")
print(f"\nFeatures ({len(X.columns)}):")
for i, col in enumerate(X.columns, 1):
    print(f"  {i:2d}. {col}")

---
## 5. Models & Evaluation

Training with **5-fold stratified cross-validation**, evaluated by **AUC**.

In [ ]:
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

def train_with_cv(model, X, y, X_test, model_name):
    """Stratified CV training. Returns OOF predictions and test predictions."""
    oof_preds  = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    auc_scores = []

    print(f"Training {model_name}...")
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model.fit(X_tr, y_tr)
        oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
        test_preds += model.predict_proba(X_test)[:, 1] / N_FOLDS

        fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
        auc_scores.append(fold_auc)
        print(f"  Fold {fold+1}: AUC = {fold_auc:.4f}")

    oof_auc = roc_auc_score(y, oof_preds)
    print(f"\n{'='*50}")
    print(f"  {model_name} — OOF AUC: {oof_auc:.4f}")
    print(f"  Mean folds: {np.mean(auc_scores):.4f} +/- {np.std(auc_scores):.4f}")
    print(f"{'='*50}\n")

    return oof_preds, test_preds, oof_auc

### 5.1 Random Forest

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=10,
    max_features='sqrt',
    class_weight='balanced',
    random_state=SEED,
    n_jobs=-1
)

oof_rf, test_preds_rf, auc_rf = train_with_cv(rf_model, X, y, X_test, "Random Forest")

### 5.2 Gradient Boosting

In [ ]:
gb_model = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    random_state=SEED
)

oof_gb, test_preds_gb, auc_gb = train_with_cv(gb_model, X, y, X_test, "Gradient Boosting")

### 5.3 LightGBM

In [ ]:
if LGBM_AVAILABLE:
    lgbm_model = lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.03,
        num_leaves=31,
        min_child_samples=20,
        colsample_bytree=0.8,
        subsample=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        class_weight='balanced',
        random_state=SEED,
        verbose=-1
    )
    oof_lgbm, test_preds_lgbm, auc_lgbm = train_with_cv(lgbm_model, X, y, X_test, "LightGBM")
else:
    print("LightGBM not installed — using Gradient Boosting as fallback")
    oof_lgbm = oof_gb.copy()
    test_preds_lgbm = test_preds_gb.copy()
    auc_lgbm = auc_gb

### 5.4 Optuna — LightGBM Tuning

50 Bayesian trials over key hyperparameters, evaluated by OOF AUC.


In [ ]:
def lgbm_objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 300, 1200),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 20, 200),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'max_depth':         trial.suggest_int('max_depth', 3, 10),
        'class_weight': 'balanced',
        'random_state': SEED,
        'verbose': -1,
        'n_jobs': -1
    }
    model = lgb.LGBMClassifier(**params)
    oof = np.zeros(len(X))
    for tr_idx, val_idx in skf.split(X, y):
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        oof[val_idx] = model.predict_proba(X.iloc[val_idx])[:, 1]
    return roc_auc_score(y, oof)

study_lgbm = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
study_lgbm.optimize(lgbm_objective, n_trials=50, show_progress_bar=True)

print(f"\nBest LightGBM AUC: {study_lgbm.best_value:.4f}")
print("Best params:")
for k, v in study_lgbm.best_params.items():
    print(f"  {k}: {v}")


In [ ]:
lgbm_tuned = lgb.LGBMClassifier(
    **study_lgbm.best_params,
    class_weight='balanced',
    random_state=SEED,
    verbose=-1,
    n_jobs=-1
)
oof_lgbm_tuned, test_preds_lgbm_tuned, auc_lgbm_tuned = train_with_cv(
    lgbm_tuned, X, y, X_test, "LightGBM (Tuned)"
)


### 5.5 XGBoost


In [ ]:
if XGB_AVAILABLE:
    xgb_model = xgb.XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        scale_pos_weight=(y == 0).sum() / (y == 1).sum(),
        random_state=SEED,
        eval_metric='auc',
        verbosity=0,
        n_jobs=-1
    )
    oof_xgb, test_preds_xgb, auc_xgb = train_with_cv(
        xgb_model, X, y, X_test, "XGBoost"
    )
else:
    print("XGBoost not installed — falling back to LightGBM Tuned.")
    oof_xgb          = oof_lgbm_tuned.copy()
    test_preds_xgb   = test_preds_lgbm_tuned.copy()
    auc_xgb          = auc_lgbm_tuned


### 5.6 Model Comparison & ROC Curves

In [ ]:
results = pd.DataFrame({
    'Model': ['Random Forest', 'Gradient Boosting', 'LightGBM',
              'LightGBM (Tuned)', 'XGBoost'],
    'OOF AUC': [auc_rf, auc_gb, auc_lgbm, auc_lgbm_tuned, auc_xgb]
}).sort_values('OOF AUC', ascending=False)

print("Model Comparison (OOF AUC):")
print(results.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

palette = ['#2ECC71', '#3498DB', '#9B59B6', '#E67E22', '#E74C3C']
axes[0].barh(results['Model'], results['OOF AUC'], color=palette[:len(results)])
axes[0].set_xlim(0.7, 1.0)
axes[0].set_xlabel('AUC')
axes[0].set_title('AUC by Model', fontsize=12)
for i, v in enumerate(results['OOF AUC']):
    axes[0].text(v + 0.001, i, f'{v:.4f}', va='center', fontweight='bold')

for oof, name, color in [
    (oof_rf,          'Random Forest',     '#2ECC71'),
    (oof_gb,          'Gradient Boosting', '#3498DB'),
    (oof_lgbm,        'LightGBM',          '#9B59B6'),
    (oof_lgbm_tuned,  'LightGBM (Tuned)',  '#E67E22'),
    (oof_xgb,         'XGBoost',           '#E74C3C'),
]:
    fpr, tpr, _ = roc_curve(y, oof)
    axes[1].plot(fpr, tpr, label=f'{name} (AUC={auc(fpr,tpr):.4f})', color=color, lw=2)

axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC=0.50)')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves (OOF)', fontsize=12)
axes[1].legend(loc='lower right', fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


### 5.7 Feature Importance

In [ ]:
rf_full = RandomForestClassifier(
    n_estimators=300, max_depth=6, min_samples_leaf=10,
    class_weight='balanced', random_state=SEED, n_jobs=-1
)
rf_full.fit(X, y)

importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_full.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=importances.head(20), x='Importance', y='Feature', palette='viridis')
plt.title('Top 20 Features — Random Forest', fontsize=13)
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print("Top 10 features:")
print(importances.head(10).to_string(index=False))

---
## 6. Ensemble

Weighted average of model predictions — typically outperforms any individual model.

In [ ]:
scores  = np.array([auc_rf, auc_gb, auc_lgbm, auc_lgbm_tuned, auc_xgb])
weights = scores / scores.sum()

print("Ensemble weights:")
for name, w in zip(['Random Forest', 'Gradient Boosting', 'LightGBM',
                    'LightGBM (Tuned)', 'XGBoost'], weights):
    print(f"  {name}: {w:.3f}")

oof_ensemble = (
    weights[0] * oof_rf +
    weights[1] * oof_gb +
    weights[2] * oof_lgbm +
    weights[3] * oof_lgbm_tuned +
    weights[4] * oof_xgb
)
test_preds_ensemble = (
    weights[0] * test_preds_rf +
    weights[1] * test_preds_gb +
    weights[2] * test_preds_lgbm +
    weights[3] * test_preds_lgbm_tuned +
    weights[4] * test_preds_xgb
)

auc_ensemble = roc_auc_score(y, oof_ensemble)
print(f"\nEnsemble OOF AUC: {auc_ensemble:.4f}")


---
## 7. Generate Submission File

In [ ]:
all_aucs = {
    'Random Forest':     auc_rf,
    'Gradient Boosting': auc_gb,
    'LightGBM':          auc_lgbm,
    'LightGBM (Tuned)':  auc_lgbm_tuned,
    'XGBoost':           auc_xgb,
    'Ensemble':          auc_ensemble
}

print("Final AUC Summary (OOF):")
for name, val in sorted(all_aucs.items(), key=lambda x: -x[1]):
    print(f"  {name}: {val:.4f}")

pred_final = test_preds_ensemble
print("\nUsing Ensemble predictions for submission.")


In [ ]:
output_dir = Path('output')
output_dir.mkdir(exist_ok=True)

submission = pd.read_csv(PATH / 'sample_submission.csv')
submission['Drafted'] = pred_final

submission_path = output_dir / 'submission.csv'
submission.to_csv(submission_path, index=False)

print(f"Submission saved to: {submission_path}")
submission.head()

In [ ]:
print("=== Submission Verification ===")
print(f"Columns: {list(submission.columns)}")
print(f"Rows: {len(submission)}")
print(f"Values outside [0,1]: {((submission['Drafted'] < 0) | (submission['Drafted'] > 1)).sum()}")
print(f"Mean prediction: {submission['Drafted'].mean():.4f}")

plt.figure(figsize=(8, 4))
plt.hist(submission['Drafted'], bins=40, edgecolor='white', color='#3498DB')
plt.xlabel('Predicted draft probability')
plt.ylabel('Player count')
plt.title('Prediction Distribution — Test Set', fontsize=13)
plt.axvline(0.5, color='red', linestyle='--', label='Threshold 0.5')
plt.legend()
plt.tight_layout()
plt.show()

---
## 8. Next Steps

### Preprocessing
- Try `KNNImputer` for smarter missing value imputation
- Scale numeric features for linear models

### Feature Engineering
- Feature interactions (e.g. `Vertical_Jump * Weight`)
- Group `School` by conference (ACC, SEC, Big Ten...)
- Position-relative performance percentiles
- Log / sqrt transforms for skewed features

### Models
- **XGBoost**: strong alternative to LightGBM
- **CatBoost**: great with categorical features
- Hyperparameter tuning with **Optuna** or `GridSearchCV`

### Advanced Strategies
- **Stacking**: use OOF predictions as features for a meta-model
- Analyze misclassifications to find patterns in errors
- More CV folds (e.g. 10)